# Day 2.2 — Keyword Search Baseline
Before reaching for embeddings, build the simplest retriever that can possibly work:
count how many words a question and a chunk share.

```text
question words -> overlap with each chunk -> rank chunks
```

A baseline is what later complexity has to beat. It also fails in a very specific way,
and seeing that failure is the point of this notebook.


## Before you begin

### Learning outcomes

- Build an explainable lexical retriever in about ten lines.
- Watch it rank the right chunk first for one question and score the right chunk **zero**
  for a paraphrase of the same idea.

Architecture reference: [D06](../diagrams/source/day_02.md).

### Expected observation

For "What keeps running during a blackout?" the chunk that actually answers the question
shares no words with it, so keyword search gives it a score of 0.

## Concept briefing

## Establish a lexical baseline first

Keyword search is limited but valuable. It is cheap, deterministic and explainable. When
the query and document use the same words, a lexical baseline may outperform a more
complex system. It fails when the question uses a paraphrase, abbreviation or related
concept absent from the chunk.

Starting with this baseline gives semantic search something measurable to improve. If a
new embedding system is slower and no more accurate on the golden set, complexity has not
earned its place.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# This notebook makes no model calls at all - retrieval only.
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.text import STOPWORDS, content_terms

chunks = load_markdown_corpus(PROJECT_ROOT / "data" / "corpus")
print("Chunks indexed:", len(chunks))

## Step 1 — Turn text into comparable words

Two texts can only be compared after they are cut into words. `content_terms` lowercases,
splits on non-letters, trims a trailing "s" so *records* matches *record*, and drops
stopwords such as *what*, *the*, *is* which appear in every question.

In [ ]:
question = "At what temperature does charging stop with a warning?"

print("question          :", question)
print("words kept        :", sorted(content_terms(question)))
print("stopwords dropped :", sorted(word for word in question.lower().replace("?", "").split() if word in STOPWORDS))

## Step 2 — Score every chunk by shared words

The score is just the size of the overlap between the question's words and the chunk's
words. No model, no vectors, no randomness: you can recompute any score by hand.

In [ ]:
def keyword_score(question, chunk):
    """How many meaningful words do the question and this chunk share?"""
    return len(content_terms(question) & content_terms(chunk.searchable_text))

def keyword_ranking(question):
    """All 15 chunks, best score first."""
    scored = [(keyword_score(question, chunk), chunk) for chunk in chunks]
    return sorted(scored, key=lambda pair: pair[0], reverse=True)

def show_top(question, k=3):
    print("Q:", question)
    for position, (score, chunk) in enumerate(keyword_ranking(question)[:k], start=1):
        print(f"  rank {position}  score {score:2}  {chunk.chunk_id:42} {chunk.section}")

show_top(question)

## Step 3 — Where the baseline wins

The question above borrowed the document's own vocabulary (*temperature*, *charging*,
*warning*), so the correct section is ranked first. Print the shared words to see why.

In [ ]:
best = keyword_ranking(question)[0][1]
shared = content_terms(question) & content_terms(best.searchable_text)

print("top chunk    :", best.chunk_id)
print("shared words :", sorted(shared))
print()
print("Nothing was learned or predicted here - the ranking is pure word overlap,")
print("which is why keyword search is cheap, instant and easy to explain to an auditor.")

## Step 4 — Break it with a paraphrase

Now ask about the same fact in ordinary English. A blackout is what the documents call
*islanded operation*, and the loads that keep running are the *priority 1 loads*. The
words differ completely, so watch the ranking collapse.

In [ ]:
PARAPHRASE = "What keeps running during a blackout?"
EXPECTED_CHUNK = "solar_microgrid:load-priorities"   # the section that really answers it

show_top(PARAPHRASE)

ranking = keyword_ranking(PARAPHRASE)
positions = [chunk.chunk_id for _, chunk in ranking]
scores = {chunk.chunk_id: score for score, chunk in ranking}

print()
print("Top score anywhere in the corpus:", ranking[0][0])
if ranking[0][0] == 0:
    print("Every chunk scores 0, so the ranking above is meaningless -")
    print("the order is just the order the chunks were loaded in.")
print()
print("Expected chunk    :", EXPECTED_CHUNK)
print("Its keyword score :", scores[EXPECTED_CHUNK])
print("Its rank          :", positions.index(EXPECTED_CHUNK) + 1, "of", len(chunks))

## Step 5 — Why it failed

Print the question's words next to the expected chunk's words. The intersection is empty:
the retriever is not "slightly wrong", it has no signal at all. Remember this question -
notebook 03 asks the identical one with a different representation.

In [ ]:
expected = next(chunk for chunk in chunks if chunk.chunk_id == EXPECTED_CHUNK)

print("question words :", sorted(content_terms(PARAPHRASE)))
print("chunk words    :", sorted(content_terms(expected.searchable_text))[:12], "...")
print("shared words   :", sorted(content_terms(PARAPHRASE) & content_terms(expected.searchable_text)))
print()
print("The chunk that answers the question:")
print(" ", expected.text)

### Try it yourself

Predict what happens if you rewrite the question using the documents' own vocabulary.
Then run the worked solution and compare the rank of the expected chunk before and after.

In [ ]:
# --- Worked solution ---
# Same information need, expressed in the words the document actually uses.
REWORDED = "Which loads have priority, and which are shed first?"

before = [chunk.chunk_id for _, chunk in keyword_ranking(PARAPHRASE)].index(EXPECTED_CHUNK) + 1
after = [chunk.chunk_id for _, chunk in keyword_ranking(REWORDED)].index(EXPECTED_CHUNK) + 1

show_top(REWORDED)
print()
print("Rank of", EXPECTED_CHUNK)
print("  with the paraphrase :", before)
print("  with document words :", after)
print()
print("Keyword search does not need better software here - it needs the user to already")
print("know the document's vocabulary. That is exactly what we cannot assume.")

### Checkpoint

**1. Why did the correct chunk score 0 for "What keeps running during a blackout?"**

<details><summary>Show answer</summary>

Because scoring is word overlap and the chunk contains none of *keeps*, *running* or
*blackout*; it says *priority 1 loads*, *emergency lighting*, *shed*. Zero shared words
means zero score, no matter how relevant the passage is to a human reader.

</details>

**2. Should we now throw keyword search away?**

<details><summary>Show answer</summary>

No. It is instant, needs no model, and is unbeatable for exact strings: error codes,
part numbers, `request_island_mode`, a section title. Production systems commonly run
lexical and semantic retrieval together (hybrid search). We are adding a second signal,
not replacing the first.

</details>

### Recap

- **Limitation we saw:** the paraphrase "What keeps running during a blackout?" gives the
  correct chunk a score of 0 - a total miss, not a near miss.
- **Layer we added:** a transparent lexical baseline whose every score can be recomputed
  by hand.
- **Evidence it worked:** the same retriever ranks the correct chunk first when the
  question reuses the document's vocabulary.